In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import torch
torch.cuda.is_available()

True

In [7]:
import torch
from package.encoder_block import TokenEmbedding, EncoderBlock

vocab_size = 100
d_model = 16
n_heads = 2
max_len = 20

# simulate a batch of 2 sequences, each with 5 tokens
token_ids = torch.tensor([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10]
], dtype=torch.long)  # (batch=2, seq_len=5)

embedding = TokenEmbedding(vocab_size, d_model)
block = EncoderBlock(d_model, n_heads, max_len)

# token ids → embeddings → encoder block
x = embedding(token_ids)  # (2, 5, 16)
out = block(x)             # (2, 5, 16)

print(x.shape)   # torch.Size([2, 5, 16])
print(out.shape) # torch.Size([2, 5, 16])


torch.Size([2, 5, 16])
torch.Size([2, 5, 16])


In [8]:
x

tensor([[[-1.3036,  1.7275,  2.3097, -1.3219,  0.6069,  0.2429,  1.0173,
           1.3095,  0.5229, -0.5247, -1.7036, -0.4326, -0.1456, -0.5182,
          -1.6271, -1.2005],
         [-0.8389,  0.9272, -0.0582, -1.6705,  0.4520,  0.2373,  0.4120,
          -1.1358,  0.2304,  1.5081,  0.7829,  0.2928,  0.3780,  0.3446,
          -0.6639,  0.8901],
         [-1.1227,  1.5561,  1.8084,  0.2911,  0.7160,  2.2183,  0.4312,
          -0.7231,  0.9326, -0.6145, -0.0728, -0.6961,  1.4605, -1.5384,
          -0.8467,  0.8267],
         [-0.4396,  0.1679, -0.5710, -1.3276,  0.1206,  0.5173,  0.6165,
          -0.2778, -0.0449,  0.0647,  1.8773, -0.2155, -0.0682, -0.8006,
          -1.8604,  0.3559],
         [ 0.9512, -1.2432, -1.1067,  0.5825, -0.7137, -0.2085,  0.1577,
          -0.9532, -0.2882, -0.8637,  0.8959,  1.2183, -0.0772,  0.9838,
           0.5046, -2.0219]],

        [[ 1.1653, -0.3375,  0.4930, -1.2373,  0.1595, -0.4023,  0.0366,
          -0.1688, -1.3950, -1.1194, -2.1587,  0.5

In [9]:
out

tensor([[[-1.2202,  1.2018,  1.8985, -0.8981,  0.5317,  0.1045,  1.0491,
           1.4926,  0.6219, -0.3802, -1.2802, -0.5029, -0.5983, -0.1638,
          -1.4296, -0.4268],
         [-1.7825,  0.7539,  0.1870, -1.8559,  0.0032,  0.0302,  0.5390,
          -0.9927,  0.5944,  1.8488,  0.5629, -0.5377, -0.2423,  0.5983,
          -1.0067,  1.3001],
         [-1.7495,  0.9716,  1.7426,  0.0360,  0.2224,  1.4340,  0.1626,
          -0.5262,  0.6451, -0.5671, -0.1241, -1.1943,  0.5202, -1.2197,
          -1.3056,  0.9518],
         [-1.0870,  0.1686,  0.0609, -1.1112, -0.0293,  0.6585,  0.9318,
           0.2421,  0.3107,  0.8102,  2.1841, -0.8190, -0.4914, -0.4388,
          -2.2110,  0.8207],
         [ 0.7073, -1.5692, -1.1545,  0.9913, -1.3268,  0.4144,  0.3766,
          -0.2087,  0.3124, -0.2389,  1.1790,  0.9351, -0.2665,  1.3152,
           0.5529, -2.0195]],

        [[ 1.2492, -0.1110,  1.3279, -0.6902,  0.7996, -0.1543,  0.2299,
          -0.0605, -1.0324, -0.3996, -1.7653,  0.2

In [13]:
import torch
from package.decoder_block import DecoderBlock

# hyperparams
d_model  = 64
n_heads  = 4
max_len  = 50
batch    = 2
src_len  = 10  # encoder sequence length
tgt_len  = 7   # decoder sequence length

# fake encoder output — what EncoderBlock.forward would have produced
enc_out = torch.randn(batch, src_len, d_model)

# fake decoder input — e.g. target tokens already embedded
x = torch.randn(batch, tgt_len, d_model)

# padding masks — True means "this position is padding, ignore it"
# last token in each sequence is padding
src_pad_mask = torch.zeros(batch, src_len, dtype=torch.bool)
src_pad_mask[:, -1] = True

tgt_pad_mask = torch.zeros(batch, tgt_len, dtype=torch.bool)
tgt_pad_mask[:, -1] = True

block = DecoderBlock(d_model, n_heads, max_len)
out = block(x, enc_out, tgt_pad_mask, src_pad_mask)

print(out.shape)  # expect (2, 7, 64)


torch.Size([2, 7, 64])


In [14]:
import torch
from package.encoder_block import EncoderBlock
from package.decoder_block import DecoderBlock

d_model = 64
n_heads = 4
max_len = 50
batch   = 2
src_len = 10
tgt_len = 7

encoder = EncoderBlock(d_model, n_heads, max_len)
decoder = DecoderBlock(d_model, n_heads, max_len)

src = torch.randn(batch, src_len, d_model)  # encoder input
tgt = torch.randn(batch, tgt_len, d_model)  # decoder input

src_pad_mask = torch.zeros(batch, src_len, dtype=torch.bool)
tgt_pad_mask = torch.zeros(batch, tgt_len, dtype=torch.bool)

# step 1: encoder reads the source sequence
enc_out = encoder(src, src_pad_mask)        # (batch, src_len, d_model)

# step 2: decoder generates output using its own input + encoder output
out = decoder(tgt, enc_out, tgt_pad_mask, src_pad_mask)  # (batch, tgt_len, d_model)

print(enc_out.shape)  # (2, 10, 64)
print(out.shape)      # (2, 7, 64)


torch.Size([2, 10, 64])
torch.Size([2, 7, 64])


In [15]:
import torch
from package.insurance_decoder_block import InsuranceDecoderBlock

batch       = 2
seq_len     = 6   # 6 past purchases
num_policies = 50
d_model     = 64
n_heads     = 4
max_len     = 50

model = InsuranceDecoderBlock(num_policies, d_model, n_heads, max_len)

policy = torch.randint(0, num_policies, (batch, seq_len))  # (2, 6)
age    = torch.randn(batch, seq_len, 1)                    # (2, 6, 1)
price  = torch.randn(batch, seq_len, 1)                    # (2, 6, 1)

policy_logits, age_pred, price_pred = model(policy, age, price)

print(policy_logits.shape)  # (2, 6, 50)
print(age_pred.shape)       # (2, 6, 1)
print(price_pred.shape)     # (2, 6, 1)


torch.Size([2, 6, 50])
torch.Size([2, 6, 1])
torch.Size([2, 6, 1])


In [19]:
# targets are the next event in the sequence
target_policy = policy[:, 1:]          # (batch, seq_len-1)
target_age    = age[:, 1:]             # (batch, seq_len-1, 1)
target_price  = price[:, 1:]          # (batch, seq_len-1, 1)

# predictions come from all positions except the last
pred_policy = policy_logits[:, :-1]   # (batch, seq_len-1, num_policies)
pred_age    = age_pred[:, :-1]        # (batch, seq_len-1, 1)
pred_price  = price_pred[:, :-1]      # (batch, seq_len-1, 1)


In [26]:
import torch
from package.insurance_decoder_block import InsuranceDecoderBlock

# ── Policy vocabulary ──────────────────────────────────────────────────────────
POLICIES = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]
policy_to_id = {p: i for i, p in enumerate(POLICIES)}  # {"A": 0, "B": 1, ...}

# ── Two customers' purchase histories ─────────────────────────────────────────
#
# Customer 1:
#   age 25 → bought policy A for 100
#   age 27 → bought policy C for 200
#   age 30 → bought policy A for 150
#
# Customer 2:
#   age 40 → bought policy B for 300
#   age 42 → bought policy D for 400
#   age 45 → bought policy B for 350

customer1_policies = ["A", "C", "A"]
customer1_ages     = [25.0, 27.0, 30.0]
customer1_prices   = [100.0, 200.0, 150.0]

customer2_policies = ["B", "D", "B"]
customer2_ages     = [40.0, 42.0, 45.0]
customer2_prices   = [300.0, 400.0, 350.0]

# ── Convert to tensors ─────────────────────────────────────────────────────────
policy = torch.tensor([
    [policy_to_id[p] for p in customer1_policies],
    [policy_to_id[p] for p in customer2_policies],
])  # (2, 3)

age = torch.tensor([
    customer1_ages,
    customer2_ages,
]).unsqueeze(-1)  # (2, 3, 1)

price = torch.tensor([
    customer1_prices,
    customer2_prices,
]).unsqueeze(-1)  # (2, 3, 1)

# ── Forward pass ──────────────────────────────────────────────────────────────
model = InsuranceDecoderBlock(num_policies=10, d_model=64, n_heads=4, max_len=50)
model.eval()

with torch.no_grad():
    policy_logits, age_pred, price_pred = model(policy, age, price)

# ── Interpret the output ───────────────────────────────────────────────────────
# position i predicts what happens at event i+1
# so we read predictions from the last position [-1] to forecast the NEXT purchase

id_to_policy = {i: p for p, i in policy_to_id.items()}

for customer_idx, name in enumerate(["Customer 1", "Customer 2"]):
    next_policy_id = policy_logits[customer_idx, -1].argmax().item()
    next_policy    = id_to_policy[next_policy_id]
    next_age       = age_pred[customer_idx, -1].item()
    next_price     = price_pred[customer_idx, -1].item()

    print(f"{name}:")
    print(f"  history : {customer1_policies if customer_idx == 0 else customer2_policies}")
    print(f"  next policy predicted : {next_policy}")
    print(f"  next age predicted    : {next_age:.1f}")
    print(f"  next price predicted  : {next_price:.1f}")
    print()


Customer 1:
  history : ['A', 'C', 'A']
  next policy predicted : E
  next age predicted    : 0.1
  next price predicted  : -0.4

Customer 2:
  history : ['B', 'D', 'B']
  next policy predicted : I
  next age predicted    : 0.1
  next price predicted  : -0.5

